Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Create the AgentCore Registry

This notebook creates a managed metadata catalog (AgentCore Registry) with **Cognito JWT authentication** and a **manual approval workflow**.

The Registry acts as the enterprise catalog for tools and agents. By configuring it with `CUSTOM_JWT` auth backed by the workshop Cognito User Pool, every token-bearing request — whether from the Gateway or from a notebook — is validated against the same identity provider.

Manual approval (`autoApproval: false`) ensures that published records require an explicit admin sign-off before they become discoverable, demonstrating enterprise governance controls.

In [ ]:
%pip install "boto3==1.42.87" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Step 1: Environment Setup

In [ ]:
import boto3, json, time

region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
sts = boto3.client("sts", region_name=region)

# Read CFN exports
cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

POOL_ID = exports["workshop-CognitoUserPoolId"]
COGNITO_DOMAIN = exports["workshop-CognitoDomain"]
M2M_CLIENT_ID = exports["workshop-CognitoM2MClientId"]
ADMIN_ROLE_ARN = exports["ac-RegistryAdminRoleArn"]

print(f"Region:         {region}")
print(f"Cognito Pool:   {POOL_ID}")
print(f"Cognito Domain: {COGNITO_DOMAIN}")
print(f"Admin Role:     {ADMIN_ROLE_ARN}")

## Step 2: Assume Admin Persona

The admin persona has full CRUD + approval permissions on the Registry.

In [ ]:
admin_creds = sts.assume_role(
    RoleArn=ADMIN_ROLE_ARN,
    RoleSessionName="workshop-admin"
)["Credentials"]

admin_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
    aws_access_key_id=admin_creds["AccessKeyId"],
    aws_secret_access_key=admin_creds["SecretAccessKey"],
    aws_session_token=admin_creds["SessionToken"],
)
print("Assumed admin persona successfully")

## Step 3: Create the Registry

We create a Registry with:
- **CUSTOM_JWT** auth pointing to the Cognito User Pool (same pool used by the Gateway)
- **Manual approval** (`autoApproval: false`) so we can demonstrate the approval workflow

In [ ]:
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{POOL_ID}/.well-known/openid-configuration"

try:
    resp = admin_client.create_registry(
        name="workshop-registry",
        description="Enterprise tool and agent catalog for the AgentCore workshop platform",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": discovery_url,
                "allowedClients": [M2M_CLIENT_ID]
            }
        },
        approvalConfiguration={"autoApproval": False},
    )
    REGISTRY_ARN = resp["registryArn"]
    REGISTRY_ID = resp.get("registryId") or REGISTRY_ARN.split("/")[-1]
    print(f"Registry created!")
    print(f"  ID:  {REGISTRY_ID}")
    print(f"  ARN: {REGISTRY_ARN}")
except admin_client.exceptions.ConflictException:
    # Registry already exists — find it
    registries = admin_client.list_registries().get("registries", [])
    existing = [r for r in registries if r["name"] == "workshop-registry"]
    if existing:
        REGISTRY_ARN = existing[0]["registryArn"]
        REGISTRY_ID = existing[0].get("registryId") or REGISTRY_ARN.split("/")[-1]
        print(f"Registry already exists:")
        print(f"  ID:  {REGISTRY_ID}")
        print(f"  ARN: {REGISTRY_ARN}")
    else:
        raise RuntimeError("Registry creation failed and no existing registry found")

## Step 4: Wait for READY Status

The Registry needs a few seconds to initialize.

In [ ]:
for attempt in range(20):
    registry = admin_client.get_registry(registryId=REGISTRY_ID)
    status = registry.get("status", "UNKNOWN")
    print(f"  Attempt {attempt + 1}: {status}")
    if status == "READY":
        print(f"\nRegistry is ready!")
        break
    time.sleep(5)
else:
    raise RuntimeError(
        f"Registry {REGISTRY_ID} did not reach READY in 100s (last status: {status}). "
        f"Check the AgentCore console for the registry state."
    )

print(f"\nRegistry ID:  {REGISTRY_ID}")
print(f"Registry ARN: {REGISTRY_ARN}")

## Summary

You created an AgentCore Registry with:

| Setting | Value | Why |
|---|---|---|
| `authorizerType` | `CUSTOM_JWT` | Same Cognito pool authenticates both Registry searches and Gateway tool calls |
| `autoApproval` | `false` | Records must be explicitly approved — demonstrates enterprise governance |
| `discoveryUrl` | Cognito OIDC endpoint | Registry validates JWT tokens from Cognito |

**Next:** Notebook 04 registers MCP tool records in the Registry.